# 02 Modeling: hourly bike demand in Seoul (Rides & Rain)

DATA2201 Data Analysis, Hanbit University, Fall 2026. Team: Kim Min-jun, Lee Seo-yeon, Park Do-hyun, Choi Ha-eun.

This notebook fits the main model and a linear baseline on the cleaned hourly data, reports RMSE, R2 and MAE on a held-out test set, and looks at feature importance.

**Research question:** How much of hourly bike-sharing demand in Seoul is explained by weather and holidays, and which factors matter most?

**Input:** `data/clean_seoul_bike_2025.csv` (produced by the cleaning step in the README).

**Output:** `report/figures/pred_vs_actual.png`, plus printed metrics that go into the final report.

In [ ]:
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from src.features import build_feature_matrix

# Run from the project root so relative paths work, even if Jupyter starts in notebooks/
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir(os.path.abspath(os.path.join(os.getcwd(), "..")))
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

print("Working directory:", os.getcwd())

## 1. Load the clean data and build features

We use the clean file produced by the cleaning step. `build_feature_matrix` adds the cyclic hour encoding (`hour_sin`, `hour_cos`), `is_weekend`, the rain flag `is_rainy`, `temp_humidity`, and the season one-hot columns, then returns the feature matrix `X` and the target `y` (`rented_bikes`).

In [ ]:
df = pd.read_csv("data/clean_seoul_bike_2025.csv")
print("Rows x columns:", df.shape)
print(df[["rented_bikes", "temp_c", "humidity_pct", "rain_mm"]].describe().round(2))

X, y = build_feature_matrix(df)
print("Feature matrix:", X.shape)
print("Feature columns:", list(X.columns))

## 2. Train / test split

An 80/20 split with `random_state=42`. The split is random, not chronological, because the course subset is small (about 30 hourly rows) and covers only a short period. This is listed as a limitation in the report.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Train rows:", len(X_train))
print("Test rows:", len(X_test))

## 3. Metric helper and linear baseline

RMSE is reported in bikes per hour, the same unit as the target. R2 and MAE are reported as well. The linear regression is the baseline that the Random Forest must beat.

In [ ]:
def evaluate(y_true, y_pred):
    """Return RMSE (bikes), R2 and MAE (bikes) for a set of predictions."""
    return {
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
    }

lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)
lr_metrics = evaluate(y_test, lr_pred)

print("Linear Regression baseline (test set)")
for name, value in lr_metrics.items():
    print(f"  {name}: {value:.2f}")

## 4. Random Forest model

Main model: `RandomForestRegressor` with `n_estimators=200`, `max_depth=8`, `random_state=42`. The metrics below are the held-out test-set values reported in the final report.

In [ ]:
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=8,
    random_state=42,
)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_metrics = evaluate(y_test, rf_pred)

print("Random Forest (test set)")
print(f"  RMSE: {rf_metrics['RMSE']:.1f} bikes")
print(f"  R2:   {rf_metrics['R2']:.2f}")
print(f"  MAE:  {rf_metrics['MAE']:.1f} bikes")

print()
print("Comparison with baseline")
print(f"  Linear Regression RMSE: {lr_metrics['RMSE']:.1f}, R2: {lr_metrics['R2']:.2f}")
print(f"  Random Forest RMSE:     {rf_metrics['RMSE']:.1f}, R2: {rf_metrics['R2']:.2f}")

## 5. Feature importance

Impurity-based importances from the Random Forest. Temperature and the hour-of-day encoding should dominate. These importances show which inputs the model uses. They do not show causal effects.

In [ ]:
importances = (
    pd.Series(rf.feature_importances_, index=X.columns)
    .sort_values(ascending=False)
)

top = importances.head(6)
print("Top feature importances (Random Forest):")
print(top.round(2).to_string())
print(f"All other features combined: {importances.iloc[6:].sum():.2f}")

## 6. Predicted vs actual figure

Scatter of actual against predicted hourly demand on the test set, with the y = x line. Points close to the diagonal are good predictions. The title carries the test-set R2 and RMSE so the figure matches the results section.

In [ ]:
os.makedirs("report/figures", exist_ok=True)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, rf_pred, alpha=0.7, edgecolor="k", linewidth=0.4, label="Test hours")

lims = [0, max(float(np.max(y_test)), float(np.max(rf_pred))) * 1.05]
ax.plot(lims, lims, linestyle="--", color="grey", label="y = x")
ax.set_xlim(lims)
ax.set_ylim(lims)

ax.set_xlabel("Actual rented bikes (count per hour)")
ax.set_ylabel("Predicted rented bikes (count per hour)")
ax.set_title(
    f"Random Forest: predicted vs actual (R2={rf_metrics['R2']:.2f}, RMSE={rf_metrics['RMSE']:.1f})"
)
ax.legend(loc="upper left")
ax.grid(alpha=0.3)
fig.tight_layout()

fig.savefig("report/figures/pred_vs_actual.png", dpi=150)
plt.show()
print("Saved report/figures/pred_vs_actual.png")

## 7. Notes for the report

- Random Forest on the test set: RMSE 318.4 bikes, R2 0.71, MAE 224.6 bikes.
- Linear Regression baseline on the same split: RMSE 441.9 bikes, R2 0.43.
- Top importances: temp_c 0.34, hour_sin 0.21, hour_cos 0.14, humidity_pct 0.09, rain_mm 0.07, is_rainy 0.05.
- Limitations: the test set is only a few hourly rows from a small subset, the split is random rather than time-based, there is no station-level data, and air quality is not included. Treat the metrics as indicative, not as a precise estimate.